# 06 · Live demo app (Gradio)

Upload a video (or pick a prepared test video) → face clips → **two models side by side**
(`mnv2_v1` = main model, `mnv2_robust` = with the defense) → verdict at the validation threshold,
**score timeline**, **Grad-CAM** per clip. The attack menu applies the same post-processing as in
notebook 05, so you can show live how re-compression or downscaling changes the verdict.

Run SETUP and cells 1–3; cell 3 prints a public `gradio.live` link (valid 72 h) and also shows the app
inline. Cell 4 writes **backup screenshots** of every prepared demo video to `demo_backup/` in Drive.
Runtime: T4 GPU (CPU also works, ~3 s per video).

In [ ]:
#@title 0 · Setup (run after every restart)
DRIVE_ROOT = "/content/drive/MyDrive/IS665_Deepfake"  #@param {type:"string"}
REPO_URL = "https://github.com/payelsarkarr/is665-team6-deepfake-recognition"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)       # newest code from GitHub
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
import cv2
if not hasattr(cv2, "CascadeClassifier"):
    !pip -q install "opencv-python-headless<5"
    print("OpenCV 4 installed -> Runtime > Restart session, then run this cell again.")
META = "/content/FaceForensics/dataset"            # official split files
if not os.path.exists(META):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ondyari/FaceForensics.git", "/content/FaceForensics"], check=True)
import importlib, ffpp
for m in ["data", "crops", "dataset", "models", "metrics", "train", "evaluate", "download", "infer", "perturb", "xai", "robustness"]:
    importlib.reload(importlib.import_module(f"ffpp.{m}"))
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 1 · Settings
MODELS = "mnv2_v1, mnv2_robust"  #@param {type:"string"}
DEMO_DIR = f"{DRIVE_ROOT}/demo_videos"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
BACKUP_DIR = f"{DRIVE_ROOT}/demo_backup"
import importlib.util
if importlib.util.find_spec("gradio") is None:
    !pip -q install gradio

In [ ]:
#@title 2 · Load the models
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
from ffpp.infer import Detector
from ffpp.perturb import ATTACKS
dets = {m.strip(): Detector(f"{RUNS_DIR}/{m.strip()}") for m in MODELS.split(",") if m.strip()}
for n, d in dets.items():
    d.warmup(); print(f"{n}: threshold {d.threshold:.3f} ({d.threshold_source}), {d.device}")
catalog = pd.read_csv(f"{DEMO_DIR}/catalog.csv") if os.path.exists(f"{DEMO_DIR}/catalog.csv") else pd.DataFrame()
display(catalog)

In [ ]:
#@title 3 · Start the app
import gradio as gr
MAIN = list(dets)[0]

def analyze(video, attack):
    if not video:
        return "Upload or pick a video.", None, []
    res = {n: d.score(video, attack=attack, explain=(n == MAIN)) for n, d in dets.items()}
    r0 = res[MAIN]
    if r0["status"] != "ok":
        return f"No face found ({r0['status']}).", None, []
    truth = ""
    if len(catalog):
        hit = catalog[catalog.file == os.path.basename(video)]
        if len(hit): truth = f"**Ground truth:** {hit.truth.iloc[0]} ({hit.method.iloc[0]})\n\n"
    md = truth + f"**Attack:** `{attack}`\n\n| Model | P(fake) | Threshold | Verdict |\n|---|---|---|---|\n"
    for n, r in res.items():
        md += f"| {n} | {r['score']:.3f} | {r['threshold']:.3f} | **{r['verdict']}** |\n"
    md += f"\n{r0['n_clips']} clips, {r0['duration_s']:.1f} s video, analysed in {r0['t_total_ms'] / 1000:.1f} s"
    fig, ax = plt.subplots(figsize=(7, 2.6))
    for n, r in res.items():
        if "frame_scores" in r:
            ys = np.concatenate(r["frame_scores"]); ax.plot(ys, lw=1.5, label=n)
        else:
            ax.plot(np.repeat(r["clip_scores"], 16), lw=1.5, label=n)
    ax.axhline(dets[MAIN].threshold, color="k", ls="--", lw=1, label="threshold")
    ax.set_ylim(0, 1); ax.set_xlabel("frame (4 clips x 16 frames)"); ax.set_ylabel("P(fake)")
    ax.legend(fontsize=7, frameon=False); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout()
    gal = []
    for c in range(r0["n_clips"]):
        gal += [(r0["clip_faces"][c], f"clip {c + 1} @ {r0['clip_start_s'][c]} s, P(fake) {r0['clip_scores'][c]:.2f}"),
                (r0["clip_cams"][c], f"clip {c + 1}: Grad-CAM ({MAIN})")]
    return md, fig, gal

examples = [[f"{DEMO_DIR}/{f}", "none"] for f in catalog.file] if len(catalog) else None
with gr.Blocks(title="Deepfake check (IS665 Team 6)") as app:
    gr.Markdown("## Deepfake check · MobileNetV2 trained on FaceForensics++ (research prototype: needs human review)")
    with gr.Row():
        with gr.Column(scale=1):
            vid = gr.Video(label="Video")
            atk = gr.Dropdown(list(ATTACKS), value="none", label="Simulated attack (post-processing)")
            btn = gr.Button("Analyse", variant="primary")
            if examples: gr.Examples(examples, [vid, atk], label="Prepared test videos")
        with gr.Column(scale=2):
            out_md = gr.Markdown(); out_plot = gr.Plot(label="Score timeline")
            out_gal = gr.Gallery(label="Face per clip and Grad-CAM", columns=4, height=360)
    btn.click(analyze, [vid, atk], [out_md, out_plot, out_gal])
app.launch(share=True, debug=False)

In [ ]:
#@title 4 · Backup screenshots of every demo video (clean + jpeg_15 + down_0.25) -> demo_backup/
os.makedirs(BACKUP_DIR, exist_ok=True)
for f in catalog.file:
    for attack in ["none", "jpeg_15", "down_0.25"]:
        md, fig, gal = analyze(f"{DEMO_DIR}/{f}", attack)
        n = len(gal)
        sheet = plt.figure(figsize=(2.3 * max(n // 2, 1), 7.5))
        ax0 = sheet.add_subplot(3, 1, 1); ax0.axis("off")
        ax0.text(0, 1, md.replace("**", "").replace("`", ""), va="top", family="monospace", fontsize=8)
        for i, (img, cap) in enumerate(gal):
            a = sheet.add_subplot(3, max(n // 2, 1), max(n // 2, 1) + 1 + (i // 2) + (i % 2) * max(n // 2, 1))
            a.imshow(img); a.axis("off"); a.set_title(cap, fontsize=6)
        sheet.savefig(f"{BACKUP_DIR}/{os.path.splitext(f)[0]}_{attack}.png", dpi=130, bbox_inches="tight")
        fig.savefig(f"{BACKUP_DIR}/{os.path.splitext(f)[0]}_{attack}_timeline.png", dpi=130, bbox_inches="tight")
        plt.close("all")
    print("saved", f)

In [ ]:
#@title 5 · Export the laptop demo bundle (models + prepared videos) -> demo_bundle.zip
# Unzip it into the demo/ folder of the repository on your laptop, then run: python demo/app.py
import zipfile
bundle = f"{DRIVE_ROOT}/demo_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for run in dets:
        z.write(f"{RUNS_DIR}/{run}/best.pt", f"models/{run}/best.pt")
        z.write(f"{RUNS_DIR}/{run}/eval/summary.json", f"models/{run}/eval/summary.json")
    for f in os.listdir(DEMO_DIR):
        z.write(f"{DEMO_DIR}/{f}", f"videos/{f}")
print(bundle, f"{os.path.getsize(bundle) / 1e6:.0f} MB")
from google.colab import files; files.download(bundle)